In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

For context, I ended up storing all my data files locally in a folder caled "varstardata", and only downloaded the ones in the list below. If you use additional data files or store them in a different folder this code will not work.

In [ ]:
# test = fits.open('./varstardata/d124.fits') #open hdul for a bias image file
# print(test[0].header.keys)

Here are the fits files we actually need
109-117: good biases
122-125: good dome flats in V
143-147: good sky flats in V
167-176, 182-196: calibration and targets, round 1; cal2 was occulted by the dome, hence why we skipped 177-181
217-223, 225-237: targets (and cal2), round 2; 224 was obstructed by the dome, so we don't use
248-262: targets, round 3
273-282: targets, round 4 (no j092 this time because it set)

In [ ]:
bias = [] #create an empty list
dome = [] #create another empty list
sky = [] #create another empty list
for i in range(300): #this loop opens every theoretical file (or at least tries to) and stores them in the appropriate list, based on bias, dark, and flats of each filter
    try: #ensures that if the file does not exist, the program doesn't crash
        hdul = fits.open(f'./varstardata/d{i}.fits') #opens a file
        if hdul[0].header['EXPTIME'] == 0 and hdul[0].header['OBJECT'] == 'Bias': #checks to see if its a bias
            data = hdul[0].data
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1) #delete those columns
            bias.append(data) #if a bias file stores image in the bias list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'Dome flats {hdul[0].header['FILTNAM']}': #check that it is indeed a a domeflat of the right color
            data = hdul[0].data
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            dome.append(data) #if a bias file stores image in the bias list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'Twilight flats {hdul[0].header['FILTNAM']}': #check that it is indeed a a skyflat of the right color
            data = hdul[0].data
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            sky.append(data) #if a bias file stores image in the bias list
        hdul.close() #close the hdul
    except FileNotFoundError: #if the file does not exist, simply moves onto the next file
        pass  

master_bias = np.median(bias,axis=0) #to get a single master bias
dome_flat = np.median(dome,axis=0) #to get a single master dome flat
sky_flat = np.median(sky,axis=0) #to get a single master sky flat
dome_flat -= master_bias
sky_flat -= master_bias #subtract the master bias
dome_flat /= np.mean(dome_flat)
sky_flat /= np.mean(sky_flat) #normalize the flats

In [ ]:
cal1 = [] #create another empty list
cal2 = [] #create another empty list
cal4 = []
j092 = []
leo = []
j1122 = []
for i in range(300): #this loop opens every theoretical file (or at least tries to) and stores them in the appropriate list, based on bias, dark, and flats of each filter
    try: #ensures that if the file does not exist, the program doesn't crash
        hdul = fits.open(f'./varstardata/d{i}.fits') #opens a file
        if hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'cal1{hdul[0].header['FILTNAM']}': #find cal1 images
            data = hdul[0].data.astype('float64') #stores it as a float, just in case
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            data -= master_bias #subtract the bias
            data /= hdul[0].header['EXPTIME'] #divide by exposure time to get counts per sec at each pixel
            cal1.append(data) #add the image data to a layer in the empty list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'cal2':
            data = hdul[0].data.astype('float64')
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            data -= master_bias
            data /= hdul[0].header['EXPTIME']
            cal2.append(data) #add the image data to a layer in the empty list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'cal4': #check that it is indeed a a skyflat of the right color
            data = hdul[0].data.astype('float64')
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            data -= master_bias
            data /= hdul[0].header['EXPTIME']
            cal4.append(data) #add the image data to a layer in the empty list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'ASAS_J092701+1412.1': #check that it is indeed a a skyflat of the right color
            data = hdul[0].data.astype('float64')
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            data -= master_bias
            data /= hdul[0].header['EXPTIME']
            j092.append(data) #add the image data to a layer in the empty list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'AH_Leo': #check that it is indeed a a skyflat of the right color
            data = hdul[0].data.astype('float64')
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            data -= master_bias
            data /= hdul[0].header['EXPTIME']
            leo.append(data) #add the image data to a layer in the empty list
        elif hdul[0].header['FILTNAM'] == 'V' and  hdul[0].header['OBJECT'] == f'ASAS_J112210+2523.4': #check that it is indeed a a skyflat of the right color
            data = hdul[0].data.astype('float64')
            for i in range(hdul[0].header['COVER']): #find the # of overscan columns from the initial bias file we used
                data = np.delete(data,-1,axis=1)
            data -= master_bias
            data /= hdul[0].header['EXPTIME']
            j1122.append(data) #add the image data to a layer in the empty list
        hdul.close() #close the hdul
    except FileNotFoundError: #if the file does not exist, simply moves onto the next file
        pass

cal1 = np.median(cal1,axis=0)
cal1 = np.divide(cal1,sky_flat, where = sky_flat != 0)
cal2 = np.median(cal2,axis=0)
cal2 = np.divide(cal2,sky_flat, where = sky_flat != 0)
cal4 = np.median(cal4,axis=0)
cal4 = np.divide(cal4,sky_flat, where = sky_flat != 0)# this condenses each object to 1 image and then divides by the normalized sky flat
aj092 = np.median(j092[0:5],axis=0) #the index groups here group the images by time of observation
aj092 = np.divide(aj092,sky_flat, where = sky_flat != 0)
bj092 = np.median(j092[5:10],axis=0)
bj092 = np.divide(bj092,sky_flat, where = sky_flat != 0)
cj092 = np.median(j092[10:15],axis=0)
cj092 = np.divide(cj092,sky_flat, where = sky_flat != 0)
aleo = np.median(leo[0:5],axis=0)
aleo = np.divide(aleo,sky_flat, where = sky_flat != 0)
bleo = np.median(leo[5:10],axis=0)
bleo = np.divide(bleo,sky_flat, where = sky_flat != 0)
cleo = np.median(leo[10:15],axis=0)
cleo = np.divide(cleo,sky_flat, where = sky_flat != 0)
dleo = np.median(leo[15:20],axis=0)
dleo = np.divide(dleo,sky_flat, where = sky_flat != 0)
aj1122 = np.median(j1122[0:5],axis=0)
aj1122 = np.divide(aj1122,sky_flat, where = sky_flat != 0)
bj1122 = np.median(j1122[5:10],axis=0)
bj1122 = np.divide(bj1122,sky_flat, where = sky_flat != 0)
cj1122 = np.median(j1122[10:15],axis=0)
cj1122 = np.divide(cj1122,sky_flat, where = sky_flat != 0)
dj1122 = np.median(j1122[15:20],axis=0)
dj1122 = np.divide(dj1122,sky_flat, where = sky_flat != 0)

In [ ]:
fig, (ax1, ax2,ax3,ax4) = plt.subplots(1,4, figsize=(12, 10)) #sets up 4 subplot; this is purely for test purposes
ax1.imshow(aj1122,vmin=0,vmax=aj1122.mean()+2*aj1122.std(),cmap='gray')
ax2.imshow(bj1122,vmin=0,vmax=bj1122.mean()+2*bj1122.std(),cmap='gray')
ax3.imshow(cj1122,vmin=0,vmax=cj1122.mean()+2*cj1122.std(),cmap='gray')
ax4.imshow(dj1122,vmin=0,vmax=dj1122.mean()+2*dj1122.std(),cmap='gray')
plt.show()

In [ ]:
from astropy.io import fits #this just creates fits files

hdu = fits.PrimaryHDU([aj1122,bj1122,cj1122,dj1122])
hdul = fits.HDUList([hdu])
hdul.writeto('j1122.fits', overwrite=True)